# Bios WebScrapping: Notebook Explicado

Este notebook carga `Bios_WebScrapping.xlsx`, revisa su estructura y deja un archivo equivalente en CSV para trabajar más fácil.

## 1. Imports y configuración

In [ ]:
from pathlib import Path

import pandas as pd

INPUT_FILE = Path("data/Bios_WebScrapping.xlsx")
OUTPUT_FILE = Path("data/Bios_WebScrapping_equivalent.csv")

## 2. Cargar el Excel

Primero validamos que el archivo exista y luego lo cargamos.

In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {INPUT_FILE}")

xl = pd.ExcelFile(INPUT_FILE)
print("Hojas disponibles:", xl.sheet_names)

df = pd.read_excel(INPUT_FILE, sheet_name=xl.sheet_names[0])
print(df.shape)
display(df.head(10))

## 3. Limpiar columnas clave

Aquí normalizamos los nombres y construimos una columna `name` que prioriza `PName` cuando viene bien, y si no usa `PName_original`.

In [ ]:
work = df.copy()

for col in ["PName_original", "PName", "iso3", "Position"]:
    if col in work.columns:
        work[col] = work[col].fillna("").astype(str).str.strip()

work["name"] = work["PName"].where(work["PName"] != "", work["PName_original"])
work = work[work["name"] != ""].copy()

display(work[["PName_original", "PName", "name", "iso3", "Position"]].head(20))

## 4. Dejar un archivo equivalente en CSV

Aquí exportamos el contenido completo del Excel a CSV para tener un equivalente plano y fácil de usar.

In [ ]:
work.to_csv(OUTPUT_FILE, index=False)
print(f"CSV guardado en: {OUTPUT_FILE}")

## 5. Ver el CSV final

Releemos el CSV exportado para comprobar que quedó bien.

In [ ]:
saved_df = pd.read_csv(OUTPUT_FILE)
print(saved_df.shape)
display(saved_df.head(20))

## 6. Si quieres solo nombres únicos

Esta celda deja una lista más compacta, con `name` e `iso3`, sin duplicados.

In [ ]:
unique_names = (
    work[["name", "iso3"]]
    .drop_duplicates()
    .sort_values(["iso3", "name"])
    .reset_index(drop=True)
)

print(unique_names.shape)
display(unique_names.head(30))